# S&P 500 Next‑Day Direction Predictor (Random Forest)

This notebook builds and evaluates a simple supervised ML model that predicts whether the **S&P 500** will close **up** or **down** on the next trading day.

**Key ideas**
- Download historical data for **^GSPC** via **yfinance** (cached to `sp500.csv` for speed).
- Create a binary label `Target` using next‑day price movement.
- Train a baseline **RandomForestClassifier** and evaluate with a **rolling backtest** (time‑series safe).
- Engineer multi‑horizon features (price relative to moving averages + recent trend) and re‑evaluate.

> **Disclaimer:** This is an educational project and **not** financial advice.

---

## How to run
1. Install dependencies: `pip install yfinance pandas scikit-learn matplotlib`
2. Run the notebook top‑to‑bottom.
3. The first run downloads data and writes `sp500.csv`; subsequent runs reuse the cached file.


## 1. Imports & data loading
Load dependencies and fetch historical data (with local CSV caching).


In [ ]:
import yfinance as yf
import pandas as pd
import os

In [ ]:
if os.path.exists("sp500.csv"):
    sp500 = pd.read_csv("sp500.csv", index_col=0)
else:
    sp500 = yf.Ticker("^GSPC")
    sp500 = sp500.history(period="max")
    sp500.to_csv("sp500.csv")

In [ ]:
sp500.index = pd.to_datetime(sp500.index)

In [ ]:
sp500

## 2. Quick visual check
Plot the S&P 500 close price to sanity‑check the dataset.


In [ ]:
sp500.plot.line(y="Close", use_index=True)

In [ ]:
del sp500["Dividends"]
del sp500["Stock Splits"]

## 3. Define the prediction target
Create:
- `Tomorrow`: next day’s close (shifted by -1)
- `Target`: 1 if tomorrow’s close is greater than today’s close, else 0


In [ ]:
sp500["Tomorrow"] = sp500["Close"].shift(-1)

In [ ]:
sp500["Target"] = (sp500["Tomorrow"] > sp500["Close"]).astype(int)

In [ ]:
sp500 = sp500.loc["1990-01-01":].copy()

In [ ]:
sp500

## 4. Baseline model
Train a baseline Random Forest on a small set of raw OHLCV features and evaluate on a holdout split.


In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, min_samples_split=100, random_state=1)

train = sp500.iloc[:-100]
test = sp500.iloc[-100:]

predictors = ["Close", "Volume", "Open", "High", "Low"]
model.fit(train[predictors], train["Target"])

In [ ]:
from sklearn.metrics import precision_score

preds = model.predict(test[predictors])
preds = pd.Series(preds, index=test.index)
precision_score(test["Target"], preds)

In [ ]:
# label the predictions with the actual price movement
combined = pd.concat([test["Target"], preds], axis=1)
combined.columns = ["Target", "Predictions"]
combined.plot()

In [ ]:
combined

## 5. Helper: prediction function
Wrap training + inference into a reusable function that returns a tidy `Target` vs `Predictions` dataframe.


In [ ]:
# Create a function to make predictions and label them with the actual price movement
def predict(train, test, predictors, model):
    model.fit(train[predictors], train["Target"])
    preds = model.predict(test[predictors])
    preds = pd.Series(preds, index=test.index, name="Predictions")
    combined = pd.concat([test["Target"], preds], axis=1)
    combined.columns = ["Target", "Predictions"]
    return combined

## 6. Rolling backtest
Evaluate the model using a walk‑forward (rolling window) backtest to avoid look‑ahead bias.


In [ ]:
# Backtest the model using a rolling window approach
def backtest(data, model, predictors, start=2500, step=250):
    all_predictions = []

    for i in range(start, data.shape[0], step):
        train = data.iloc[0:i].copy()
        test = data.iloc[i:(i+step)].copy()
        predictions = predict(train, test, predictors, model)
        all_predictions.append(predictions)
    
    return pd.concat(all_predictions)

In [ ]:
predictions = backtest(sp500, model, predictors)

In [ ]:
predictions

In [ ]:
predictions["Predictions"].value_counts()

In [ ]:
precision_score(predictions["Target"], predictions["Predictions"])

In [ ]:
predictions["Target"].value_counts() / predictions.shape[0]

## 7. Feature engineering across multiple horizons
Create features that capture:
- **Mean reversion**: today’s close relative to a rolling mean (`Close_Ratio_*`)
- **Momentum/trend**: recent up‑day counts (`Trend_*`) using `Target` shifted by 1 day to avoid leakage


In [ ]:
horizons = [2,5,60,250,1000]
new_predictors = []

for horizon in horizons:
    rolling_averages = sp500.rolling(horizon).mean()
    
    ratio_column = f"Close_Ratio_{horizon}"
    sp500[ratio_column] = sp500["Close"] / rolling_averages["Close"]
    
    trend_column = f"Trend_{horizon}"
    sp500[trend_column] = sp500.shift(1).rolling(horizon).sum()["Target"]
    
    new_predictors+= [ratio_column, trend_column]

## 8. Final dataset cleanup
Drop early rows where rolling features are undefined (NaNs), while allowing `Tomorrow` to remain NaN on the final row.


In [ ]:
sp500 = sp500.dropna(subset=sp500.columns[sp500.columns != "Tomorrow"])

In [ ]:
sp500

## 9. Updated model & probability thresholding
Switch to probability predictions (`predict_proba`) and apply a threshold (0.6) to trade off precision vs coverage.


In [ ]:
model = RandomForestClassifier(n_estimators=200, min_samples_split=50, random_state=1)

In [ ]:
def predict(train, test, predictors, model):
    model.fit(train[predictors], train["Target"])
    preds = model.predict_proba(test[predictors])[:,1]
    preds[preds >=.6] = 1
    preds[preds <.6] = 0
    preds = pd.Series(preds, index=test.index, name="Predictions")
    combined = pd.concat([test["Target"], preds], axis=1)
    return combined

In [ ]:
predictions = backtest(sp500, model, new_predictors)

In [ ]:
predictions["Predictions"].value_counts()

In [ ]:
precision_score(predictions["Target"], predictions["Predictions"])

In [ ]:
predictions["Target"].value_counts() / predictions.shape[0]

In [ ]:
predictions

---

## Notes & possible extensions (optional)
- Try alternative models (logistic regression, gradient boosting).
- Add additional features (volatility, returns, RSI, macro variables).
- Evaluate additional metrics and trading‑style backtests (transaction costs, drawdown).
